# Jump sites with a log-normal distribution of jump rates
The model<sup>1</sup> extends the [equivalent sites on a circle](equivalent_sites_circle.ipynb) model to a disordered environment, for example methyl groups in a glassy polymer, where every group has its own barrier. The jump rates then follow a log-normal distribution of standard deviation $\sigma$, centred on the residence time $\tau$:

$$ S(Q, E) = A_0(Q)\,\delta(E) + \sum_{l=1}^{N-1} A_l(Q) \sum_{s=0}^{20} g_s\, L(E, \Gamma_l r_s), $$

with $A_l(Q)$ and $\Gamma_l$ as in the circle model. The distribution is sampled at 21 log-spaced widths $\Gamma_l r_s$, with $\ln r_s = \sigma c (s/10 - 1)$ and $c = \sqrt{-2 \ln 0.1}$, and weights $g_s \propto \exp(-\ln^2 r_s / (2\sigma^2))$ that sum to one.

$A_0$ is the Elastic Incoherent Structure Factor (EISF) and $g_s A_l$ the Quasi-Elastic Incoherent Structure Factor (QISF) of one Lorentzian. The Lorentzian of order $j = (l - 1) \cdot 21 + s + 1$ has circle order $l$ and sample $s$, so there are $21 (N - 1)$ Lorentzians per $Q$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## The model at different Q values

In [ ]:
# Q is in Angstrom^-1, the radius in Angstrom and the residence time in ps.
Q = np.linspace(0.5, 2, 4)

log_normal_model = edyn.JumpSitesLogNormalDistribution(
    scale=1.0,
    radius=1.0,
    residence_time=2.0,
    sigma=0.5,
    number_of_sites=3,
    Q=Q,
)
print(f'{log_normal_model.number_of_lorentzians} Lorentzians per Q')

In [ ]:
# Plot the quasi-elastic part (all Lorentzians) at every Q. The delta function
# would be a spike at E = 0; its weight, the EISF, is plotted further down.
energy = np.linspace(-2, 2, 501)
component_collections = log_normal_model.get_component_collections()

cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    quasielastic = sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )
    color = cmap(Q_index / (nQ - 1))
    plt.plot(energy, quasielastic, label=f'Q={Q[Q_index]:.2f} Å^-1', color=color)

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Jump sites with a log-normal distribution, sigma = 0.5')
plt.show()

## The sampled distribution

The grid of widths scales with $\sigma$, so the weights $g_s$ are the same for every $\sigma$; only the widths change.

In [ ]:
plt.figure()
plt.bar(range(21), log_normal_model.sample_weights())
plt.xlabel('Sample s')
plt.ylabel('Weight $g_s$')
plt.title('Weights of the 21 sampled widths')
plt.show()

for order in [1, 11, 21]:
    width = log_normal_model.calculate_width(order=order)[0]
    print(f'Lorentzian of order {order}: HWHM = {width:.4f} meV')

## The effect of sigma

A wider distribution gives a quasi-elastic line with a sharper peak and longer wings. For $\sigma \to 0$ the model becomes the equivalent-sites circle model.

In [ ]:
energy = np.linspace(-2, 2, 501)
Q_single = 1.0


def quasielastic(
    model: edyn.EquivalentSitesCircle | edyn.JumpSitesLogNormalDistribution,
) -> np.ndarray:
    collection = model.get_component_collections()[0]
    return sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )


circle_model = edyn.EquivalentSitesCircle(
    radius=1.0, residence_time=2.0, number_of_sites=3, Q=Q_single
)

plt.figure()
plt.plot(energy, quasielastic(circle_model), color='k', linestyle='dashed', label='circle model')
for sigma in [0.01, 0.5, 1.0]:
    model = edyn.JumpSitesLogNormalDistribution(
        radius=1.0, residence_time=2.0, sigma=sigma, number_of_sites=3, Q=Q_single
    )
    plt.plot(energy, quasielastic(model), label=f'sigma = {sigma}')
plt.yscale('log')
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Quasi-elastic part at Q = 1 Å$^{-1}$')
plt.legend()
plt.show()

The EISF does not depend on $\sigma$: the distribution changes how fast the jumps are, not where the sites are.

In [ ]:
Q_fine = np.linspace(0.01, 4, 200)
eisf = log_normal_model.calculate_EISF(Q_fine)
print('Same EISF as the circle model:', np.allclose(eisf, circle_model.calculate_EISF(Q_fine)))
print('EISF + QISF = 1:', np.allclose(eisf + log_normal_model.calculate_QISF(Q_fine), 1))

## References

1. A. Chahid, A. Alegría and J. Colmenero, Macromolecules **27**, 3282 (1994), [doi:10.1021/ma00090a022](https://doi.org/10.1021/ma00090a022).